<div align="center"> 

---



# **Cerebellum Mega Analysis**


<img src="https://static.wixstatic.com/media/da4857_5f6dbdb6bbd84328b1e31c7f2dafb673~mv2.png/v1/fill/w_224,h_226,al_c,q_85,usm_0.66_1.00_0.01,enc_auto/enigma_addiction-300x249.png" alt="image" style="width:300px;"/>


 </img>
</div>


---
# <u> **Project Description:**</u>

The aim of this study is to perform a multi-site mega/meta analysis to asses the effects of substance use on volumetric changes in cerebellar regions (and far beyond). 

## **Hypothesis to test**

**Variables to consider:** 
  - Age
  - Sex
  - Polysubstance use : Smoke and alcohol
  
  
  **Analysis:**
   1. Volume comparisons between-groups of:
   - cerebellum regions

2. Main effects: MDTB atlas
- Cerebellar volume ~ any drug
- Cerebellar volume ~ drug type

3. Interaction effects: Cerebellum volume
- group * sex
- group + age + sex

4. Partial Least Square: Cerebral - cerebellar association
- MDTB atlas

---

In [1]:
addTaskCallback(function(...) {set.seed(42);TRUE})
options(warn=-1) # to turn off warnings
#options(warn=0) # to turn on warnings
options(repr.plot.width=10, repr.plot.height=6) # to resize plots

1 
1

In [2]:
setwd("/mnt/g/Other computers/My Laptop/PhD/Psilantro/Enigma_cerebellum/Enigma_Cerebellum/")

## Load Packages

In [3]:
# Load required packages  
  pacman::p_load(tidyverse,devtools,readxl,ggbeeswarm,gtExtras,gt,webshot,webshot2,VIM,
                ggpubr,cowplot,gtsummary,ff,ggthemes,scales,mice,vtreat,sjmisc,SmartEDA,boot,MKinfer,MatchIt,
                ggcorrplot,olsrr,lme4,rstatix,janitor,coin,rcartocolor,ggdist,gghalves,ggnewscale,patchwork,
                lsr,tidyr,reshape2,broom,irr,ggpubr,caret,tidymodels,foreach,doParallel,optmatch,emmeans)


## Set settings for plotting

In [4]:
theme_output  <- theme(text = element_text(family = "Times New Roman"),
                        axis.text.x = element_text(size = 20),
                        axis.text.y = element_text(size = 20),
                        axis.title.x = element_text(size = 25, face = "bold"),
                        axis.title.y = element_text(size = 25, face = "bold"),
                        legend.text = element_text(size = 22), legend.title = element_text(size = 24),
                        title = element_text(size = 20,face = "italic"))

pal_sex <- c(alpha("#8E5A9C",1),alpha("#2879BC",1))
pal_user <- c(alpha("#757575",1),alpha("#059DC7",1))
pal_substances <- c("HC" = alpha("#757575",1), "ALC" = alpha("#C93F4D",1), 
    "ATS" = alpha("#EB8F33",1), "CANN" = alpha("#0B8755",1), 
    "COC" = alpha("#3A5589",1), "NIC" = alpha("#764EA1",1),
    "OPI" = alpha("#06AED5",1))

In [5]:
`%notin%` <- Negate(`%in%`)

## Load data

In [6]:
load("Data/RData/Volumes_Cerebellum.RData")
load("Statistics/Linear_models/MDTB/MDTB_volumes_statistics.RData")

## **1. Interclass correlation coefficient (ICC)**

Calculating the ICC between the full analysis and the median fold

### **1.1 User**

final files:  
**MDTB_user_merged_full_folds_matchit**  
**MDTB_user_icc**

#### Join all data

In [7]:
MDTB_user_merged_full_folds <- merge(p_values_MDTB_user_vs_control,
                    p_values_MDTB_folds_users_vs_control_median,
                    by = c("region"),
                    suffixes = c("", "_folds"))

MDTB_user_merged_full_folds_matchit <- merge(MDTB_user_merged_full_folds,
                    p_values_MDTB_matchit_users_vs_control,
                    by = c("region"),
                    suffixes = c("", "_matchit")) %>%
    mutate(diff_effect.size_std_FullvsMatchit = (effect.size - effect.size_matchit) / sd(effect.size - effect.size_matchit, na.rm = TRUE),
        diff_statistic_abs_FullvsMatchit = abs(statistic - statistic_matchit),
        significant_both_FullvsMatchit = ifelse(p.adj < 0.05 & p.adj_matchit < 0.05, TRUE, FALSE),
        diff_effect.size_std_FullvsFolds = (effect.size - effect.size_median) / sd(effect.size - effect.size_median, na.rm = TRUE),
        diff_statistic_abs_FullvsFolds = abs(statistic - statistic_median),
        significant_both_FullvsFolds = ifelse(p.adj < 0.05 & p_adj_median < 0.05, TRUE, FALSE),
        # Load CA data if not exists\nif (!exists("p_values_MDTB_CA_user_vs_control")) {\n    load("Statistics/Sanity_check/MDTB/MDTB_sanity_check_statistics.RData")\n}\n# Join CA data\n# NOTE: Extending significant_all from 3 to 4 strategies (Full + Folds + Matchit + CA)
    ) %>%
    left_join(
        p_values_MDTB_CA_user_vs_control %>%
            select(region, statistic, p.adj, effect.size) %>% rename(p.adj_CA = p.adj, statistic_CA = statistic, effect.size_CA = effect.size),
        by = "region"
    ) %>%
    mutate(
        significant_all = ifelse(significant_both_FullvsMatchit == TRUE & significant_both_FullvsFolds == TRUE & !is.na(p.adj_CA) & p.adj_CA < 0.05, TRUE, FALSE)
        )



#### Full vs matchID

In [8]:
MDTB_user_icc_t_value_full_vs_matchit <- icc(MDTB_user_merged_full_folds_matchit[, c("statistic", "statistic_matchit")], model = "twoway", type = "agreement", unit = "single")
MDTB_user_icc_d_value_full_vs_matchit <- icc(MDTB_user_merged_full_folds_matchit[, c("effect.size", "effect.size_matchit")], model = "twoway", type = "agreement", unit = "single")

#### Full vs folds

In [9]:
MDTB_user_icc_t_value_full_vs_folds <- icc(MDTB_user_merged_full_folds_matchit[, c("statistic", "statistic_median")], model = "twoway", type = "agreement", unit = "single")
MDTB_user_icc_d_value_full_vs_folds <- icc(MDTB_user_merged_full_folds_matchit[, c("effect.size", "effect.size_median")], model = "twoway", type = "agreement", unit = "single")

#### matchID vs folds

In [10]:
MDTB_user_icc_t_value_matchit_vs_folds <- icc(MDTB_user_merged_full_folds_matchit[, c("statistic_matchit", "statistic_median")], model = "twoway", type = "agreement", unit = "single")
MDTB_user_icc_d_value_matchit_vs_folds <- icc(MDTB_user_merged_full_folds_matchit[, c("effect.size_matchit", "effect.size_median")], model = "twoway", type = "agreement", unit = "single")

#### Merge ICCs

In [11]:
MDTB_user_icc <- list(t_value_fullvsmatchit = MDTB_user_icc_t_value_full_vs_matchit,
                        d_value_fullvsmatchit = MDTB_user_icc_d_value_full_vs_matchit,
                        t_value_fullvsfolds = MDTB_user_icc_t_value_full_vs_folds,
                        d_value_fullvsfolds = MDTB_user_icc_d_value_full_vs_folds,
                        t_value_matchitvsfolds = MDTB_user_icc_t_value_matchit_vs_folds,
                        d_value_matchitvsfolds = MDTB_user_icc_d_value_matchit_vs_folds)

### **1.2 Substance**

final files:  
**MDTB_substance_merged_full_folds_matchit**  
**MDTB_substance_icc**

#### Join all data

In [12]:
MDTB_substance_merged_full_folds <- merge(p_values_MDTB_substance_vs_control,
                    p_values_MDTB_folds_substance_vs_control_median,
                    by = c("region", "Substance"),
                    suffixes = c("", "_folds"))

MDTB_substance_merged_full_folds_matchit <- merge(MDTB_substance_merged_full_folds,
                    p_values_MDTB_matchit_substance_vs_control,
                    by = c("region", "Substance"),
                    suffixes = c("", "_matchit")) %>%
    mutate(diff_effect.size_std_FullvsMatchit = (effect.size - effect.size_matchit) / sd(effect.size - effect.size_matchit, na.rm = TRUE),
        diff_statistic_abs_FullvsMatchit = abs(statistic - statistic_matchit),
        significant_both_FullvsMatchit = ifelse(p_adj < 0.05 & p_adj_matchit < 0.05, TRUE, FALSE),
        diff_effect.size_std_FullvsFolds = (effect.size - effect.size_median) / sd(effect.size - effect.size_median, na.rm = TRUE),
        diff_statistic_abs_FullvsFolds = abs(statistic - statistic_median),
        significant_both_FullvsFolds = ifelse(p_adj < 0.05 & p_adj_median < 0.05, TRUE, FALSE),
        # CA data is already loaded from User section
    # Join CA data for Substance analysis
    ) %>%
    left_join(
        p_values_MDTB_CA_substance_vs_control %>%
            select(region, Substance, statistic, p.adj, effect.size) %>% rename(p.adj_CA = p.adj, statistic_CA = statistic, effect.size_CA = effect.size),
        by = c("region", "Substance")
    ) %>%
    mutate(
        significant_all = ifelse(significant_both_FullvsMatchit == TRUE & significant_both_FullvsFolds == TRUE & !is.na(p.adj_CA) & p.adj_CA < 0.05, TRUE, FALSE)
        )



#### Full vs matchID

In [13]:
MDTB_substance_icc_t_value_full_vs_matchit <- icc(MDTB_substance_merged_full_folds_matchit[, c("statistic", "statistic_matchit")], model = "twoway", type = "agreement", unit = "single")
MDTB_substance_icc_d_value_full_vs_matchit <- icc(MDTB_substance_merged_full_folds_matchit[, c("effect.size", "effect.size_matchit")], model = "twoway", type = "agreement", unit = "single")

#### Full vs folds

In [14]:
MDTB_substance_icc_t_value_full_vs_folds <- icc(MDTB_substance_merged_full_folds_matchit[, c("statistic", "statistic_median")], model = "twoway", type = "agreement", unit = "single")
MDTB_substance_icc_d_value_full_vs_folds <- icc(MDTB_substance_merged_full_folds_matchit[, c("effect.size", "effect.size_median")], model = "twoway", type = "agreement", unit = "single")

#### matchID vs folds

In [15]:
MDTB_substance_icc_t_value_matchit_vs_folds <- icc(MDTB_substance_merged_full_folds_matchit[, c("statistic_matchit", "statistic_median")], model = "twoway", type = "agreement", unit = "single")
MDTB_substance_icc_d_value_matchit_vs_folds <- icc(MDTB_substance_merged_full_folds_matchit[, c("effect.size_matchit", "effect.size_median")], model = "twoway", type = "agreement", unit = "single")

#### Merge ICC

In [16]:
MDTB_substance_icc <- list(t_value_fullvsmatchit = MDTB_substance_icc_t_value_full_vs_matchit,
                        d_value_fullvsmatchit = MDTB_substance_icc_d_value_full_vs_matchit,
                        t_value_fullvsfolds = MDTB_substance_icc_t_value_full_vs_folds,
                        d_value_fullvsfolds = MDTB_substance_icc_d_value_full_vs_folds,
                        t_value_matchitvsfolds = MDTB_substance_icc_t_value_matchit_vs_folds,
                        d_value_matchitvsfolds = MDTB_substance_icc_d_value_matchit_vs_folds)

## Save

In [17]:
MDTB_user_merged_full_folds_matchit %>% write_csv("Statistics/Linear_Models/MDTB/MDTB_user_vs_control.csv")
MDTB_substance_merged_full_folds_matchit %>% write_csv("Statistics/Linear_Models/MDTB/MDTB_substances_vs_control.csv")

In [18]:
save(MDTB_user_merged_full_folds_matchit,
    MDTB_user_icc,
    MDTB_substance_merged_full_folds_matchit,
    MDTB_substance_icc,
    file = "Statistics/Linear_Models/MDTB/MDTB_full_folds_matchit_comparison.RData")

---
## **Sanity Check: ICC between full analysis - match ID and complete cases - imputation datasets (CC and AI)**

Assesses how much the choice of imputation strategy changes effect sizes by computing ICC:
- **Full vs CC** (complete cases)
- **Full vs AI** (alternative imputation — no neuroanatomical predictors)
- **Match ID vs CC**
- **Match ID vs AI**


In [19]:

load("Statistics/Sanity_check/MDTB/MDTB_sanity_check_statistics.RData")

# ---- User vs HC: ICC primary vs CC and primary vs AI (MDTB) ----
MDTB_user_SC_merged <- p_values_MDTB_user_vs_control %>%
    select(region, effect.size, statistic) %>%
    inner_join(
        p_values_MDTB_CC_user_vs_control %>% select(region, effect.size, statistic),
        by = "region", suffix = c("", "_CC")
    ) %>%
    inner_join(
        p_values_MDTB_AI_user_vs_control %>% select(region, effect.size, statistic),
        by = "region", suffix = c("", "_AI")
    )

MDTB_user_icc_d_full_vs_CC <- icc(MDTB_user_SC_merged[, c("effect.size", "effect.size_CC")],
                                    model = "twoway", type = "agreement", unit = "single")
MDTB_user_icc_d_full_vs_AI <- icc(MDTB_user_SC_merged[, c("effect.size", "effect.size_AI")],
                                    model = "twoway", type = "agreement", unit = "single")

# ---- Substance vs HC: ICC primary vs CC and primary vs AI (MDTB) ----
MDTB_substance_SC_merged <- p_values_MDTB_substance_vs_control %>%
    select(region, Substance, effect.size, statistic) %>%
    inner_join(
        p_values_MDTB_CC_substance_vs_control %>% select(region, Substance, effect.size, statistic),
        by = c("region", "Substance"), suffix = c("", "_CC")
    ) %>%
    inner_join(
        p_values_MDTB_AI_substance_vs_control %>% select(region, Substance, effect.size, statistic),
        by = c("region", "Substance"), suffix = c("", "_AI")
    )

MDTB_substance_icc_d_full_vs_CC <- icc(MDTB_substance_SC_merged[, c("effect.size", "effect.size_CC")],
                                         model = "twoway", type = "agreement", unit = "single")
MDTB_substance_icc_d_full_vs_AI <- icc(MDTB_substance_SC_merged[, c("effect.size", "effect.size_AI")],
                                         model = "twoway", type = "agreement", unit = "single")

icc_sc_summary_mdtb <- tibble(
    comparison      = c("User: Primary vs CC", "User: Primary vs AI",
                        "Substance: Primary vs CC", "Substance: Primary vs AI"),
    ICC_effect_size = c(MDTB_user_icc_d_full_vs_CC$value, MDTB_user_icc_d_full_vs_AI$value,
                        MDTB_substance_icc_d_full_vs_CC$value, MDTB_substance_icc_d_full_vs_AI$value),
    ICC_lower95     = c(MDTB_user_icc_d_full_vs_CC$lbound, MDTB_user_icc_d_full_vs_AI$lbound,
                        MDTB_substance_icc_d_full_vs_CC$lbound, MDTB_substance_icc_d_full_vs_AI$lbound),
    ICC_upper95     = c(MDTB_user_icc_d_full_vs_CC$ubound, MDTB_user_icc_d_full_vs_AI$ubound,
                        MDTB_substance_icc_d_full_vs_CC$ubound, MDTB_substance_icc_d_full_vs_AI$ubound)
)

dir.create("Statistics/Sanity_check/MDTB", recursive = TRUE, showWarnings = FALSE)
write_csv(icc_sc_summary_mdtb, "Statistics/Sanity_check/MDTB/ICC_imputation_sensitivity.csv")
print(icc_sc_summary_mdtb)

save(MDTB_user_SC_merged, MDTB_substance_SC_merged, icc_sc_summary_mdtb,
     MDTB_user_icc_d_full_vs_CC, MDTB_user_icc_d_full_vs_AI,
     MDTB_substance_icc_d_full_vs_CC, MDTB_substance_icc_d_full_vs_AI,
     file = "Statistics/Sanity_check/MDTB/ICC_imputation_sensitivity.RData")


# A tibble: 4 × 4
  comparison               ICC_effect_size ICC_lower95 ICC_upper95
  <chr>                              <dbl>       <dbl>       <dbl>
1 User: Primary vs CC                0.852     -0.0279       0.975
2 User: Primary vs AI                0.993      0.244        0.999
3 Substance: Primary vs CC           0.969      0.936        0.984
4 Substance: Primary vs AI           0.998      0.996        0.999


In [20]:

load("Statistics/Sanity_check/MDTB/MDTB_sanity_check_statistics.RData")

# ---- User vs HC: ICC matchid vs CC and matchid vs AI ----
MDTB_user_SC_merged <- MDTB_user_merged_full_folds_matchit %>%
    select(region, effect.size_matchit, statistic_matchit) %>%
    inner_join(
        p_values_MDTB_CC_user_vs_control %>%
            select(region, effect.size, statistic) %>%
            rename_with(~ paste0(.x, "_CC"), -region),
        by = "region"
    ) %>%
    inner_join(
        p_values_MDTB_AI_user_vs_control %>% select(region, effect.size, statistic) %>%
            rename_with(~ paste0(.x, "_AI"), -region),
        by = "region"
    )

MDTB_user_icc_d_matchid_vs_CC <- icc(MDTB_user_SC_merged[, c("effect.size_matchit", "effect.size_CC")],
                                       model = "twoway", type = "agreement", unit = "single")
MDTB_user_icc_d_matchid_vs_AI <- icc(MDTB_user_SC_merged[, c("effect.size_matchit", "effect.size_AI")],
                                       model = "twoway", type = "agreement", unit = "single")
                                       
# ---- Substance vs HC: ICC matchid vs CC and matchid vs AI ----
MDTB_substance_SC_merged <- MDTB_substance_merged_full_folds_matchit %>%
    select(region, Substance, effect.size_matchit, statistic_matchit) %>%
    inner_join(
        p_values_MDTB_CC_substance_vs_control %>% select(region, Substance, effect.size, statistic) %>%
            rename_with(~ paste0(.x, "_CC"), -c(region, Substance)),
        by = c("region", "Substance")
    ) %>%
    inner_join(
        p_values_MDTB_AI_substance_vs_control %>% select(region, Substance, effect.size, statistic) %>%
            rename_with(~ paste0(.x, "_AI"), -c(region, Substance)),
        by = c("region", "Substance")
    )

MDTB_substance_icc_d_matchid_vs_CC <- icc(MDTB_substance_SC_merged[, c("effect.size_matchit", "effect.size_CC")],
                                            model = "twoway", type = "agreement", unit = "single")
MDTB_substance_icc_d_matchid_vs_AI <- icc(MDTB_substance_SC_merged[, c("effect.size_matchit", "effect.size_AI")],
                                            model = "twoway", type = "agreement", unit = "single")

# Summary
icc_sc_summary <- tibble(
    comparison              = c("User: MatchID vs CC", "User: MatchID vs AI",
                                "Substance: MatchID vs CC", "Substance: MatchID vs AI"),
    ICC_effect_size         = c(MDTB_user_icc_d_matchid_vs_CC$value,
                                MDTB_user_icc_d_matchid_vs_AI$value,
                                MDTB_substance_icc_d_matchid_vs_CC$value,
                                MDTB_substance_icc_d_matchid_vs_AI$value),
    ICC_lower95             = c(MDTB_user_icc_d_matchid_vs_CC$lbound,
                                MDTB_user_icc_d_matchid_vs_AI$lbound,
                                MDTB_substance_icc_d_matchid_vs_CC$lbound,
                                MDTB_substance_icc_d_matchid_vs_AI$lbound),
    ICC_upper95             = c(MDTB_user_icc_d_matchid_vs_CC$ubound,
                                MDTB_user_icc_d_matchid_vs_AI$ubound,
                                MDTB_substance_icc_d_matchid_vs_CC$ubound,
                                MDTB_substance_icc_d_matchid_vs_AI$ubound)
)

dir.create("Statistics/Sanity_check/MDTB", recursive = TRUE, showWarnings = FALSE)
write_csv(icc_sc_summary, "Statistics/Sanity_check/MDTB/ICC_imputation_sensitivity_matchid.csv")
print(icc_sc_summary)

save(MDTB_user_SC_merged, MDTB_substance_SC_merged, icc_sc_summary,
     MDTB_user_icc_d_matchid_vs_CC, MDTB_user_icc_d_matchid_vs_AI,
     MDTB_substance_icc_d_matchid_vs_CC, MDTB_substance_icc_d_matchid_vs_AI,
     file = "Statistics/Sanity_check/MDTB/ICC_imputation_sensitivity_matchid.RData")


# A tibble: 4 × 4
  comparison               ICC_effect_size ICC_lower95 ICC_upper95
  <chr>                              <dbl>       <dbl>       <dbl>
1 User: MatchID vs CC                0.456      -0.123       0.831
2 User: MatchID vs AI                0.654       0.124       0.899
3 Substance: MatchID vs CC           0.568       0.350       0.729
4 Substance: MatchID vs AI           0.588       0.372       0.743
